<a href="https://colab.research.google.com/github/jonathasArauj/FIAP_IA_TURMAB_GRUPO_BH_MG_FASE6/blob/main/Lauren_Cota_rm569789_Jonathas_Martins_de_Araujo_rm574051_Carlos_Magno_Barcelos_rm569944_pbl_fase6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FarmTech Solutions — Visão Computacional
**FIAP · Graduação em Inteligência Artificial · Fase 6**

| Integrante | RM |
|---|---:|
| Lauren Cota | 569789 |
| Jonathas Martins de Araujo | 574051 |
| Carlos Magno Barcelos | 569944 |

## Problema e solução
Demonstrar a um cliente da FarmTech como reconhecer e localizar **xícaras e tesouras** em imagens. Treinamos um YOLOv5s customizado com dois orçamentos (30 e 60 épocas) e comparamos o modelo selecionado com **YOLOv3 pré-treinado no COCO** e uma **CNN treinada do zero**.

O YOLOv5s ajusta um detector às duas classes. O YOLOv3 representa a abordagem pré-treinada, usando pesos Darknet com OpenCV DNN. A CNN classifica a imagem inteira, sem localizar o objeto; portanto, seus resultados não são diretamente equivalentes aos de detectores.

## Organização
1. Ambiente, dados e análise exploratória.
2. Entrega 1: treinamento, validação, teste e imagens detectadas.
3. Entrega 2: comparação das três abordagens.
4. Conclusões, pontos fortes e limitações.
5. Ficha técnica e resultados exportados.

## Dados e execução
O dataset contém **80 fotos**, 40 por classe, divididas em **32 treino / 4 validação / 4 teste por classe**. Os rótulos seguem o formato YOLO. O conjunto de imagens e rótulos é armazenado separadamente no arquivo `data/Fase 6 - Dataset/dataset_fase6.zip` do repositório. **Antes de executar**, baixe essa pasta e envie-a ao seu **Meu Drive** (o ZIP deve ficar em `MyDrive/Fase 6 - Dataset/dataset_fase6.zip`), como explicado no README do repositório.

Execute este notebook no **Google Colab**, selecione GPU T4 quando disponível e use **Executar tudo**. A primeira execução requer internet para instalar bibliotecas e obter os pesos necessários. A célula de preparação monta o Drive e extrai o ZIP para o disco temporário do Colab; o notebook não contém imagens embutidas e não oferece execução local sem adaptação da etapa de montagem do Drive.

As métricas, os tempos e as conclusões comparativas abaixo são **gerados automaticamente a partir dos números de cada execução** (células marcadas como "gerada automaticamente"); portanto, ao executar de novo, o texto acompanha os novos resultados, que podem variar levemente conforme hardware, dependências e ordem dos lotes. A pasta de resultados da execução registrada neste notebook é impressa na ficha técnica (seção 5).


## 1. Ambiente e dados
Instalamos as dependências, carregamos as imagens e verificamos a divisão do dataset antes do treinamento.


In [ ]:
# Prepara o ambiente: instala só o que estiver faltando no Colab
import importlib.util, subprocess, sys

# Instala o OpenCV headless (sem interface gráfica), usado para ler as imagens; a seção 2.1 instala também a versão pedida pelo YOLOv5.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless==4.10.0.84'], check=True)
# nome do módulo importado -> nome do pacote no pip
pacotes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'PIL':'pillow',
           'IPython':'ipython', 'torch':'torch', 'torchvision':'torchvision',
           'tensorflow':'tensorflow', 'yaml':'pyyaml', 'sklearn':'scikit-learn', 'tabulate':'tabulate'}
# lista os pacotes que ainda não estão instalados
faltantes = [pacote for modulo,pacote in pacotes.items() if importlib.util.find_spec(modulo) is None]
# instala de uma vez só os que faltam (evita reinstalar o que o Colab já tem)
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltantes], check=True)
print('Ambiente preparado.')

In [ ]:
# O dataset fica no Google Drive; não é incorporado ao notebook.
from pathlib import Path
import zipfile, shutil, sys

# Detecta se o notebook está rodando no Google Colab
IN_COLAB = 'google.colab' in sys.modules
# o carregamento do dataset depende do Google Drive, que só existe no Colab
if not IN_COLAB:
    raise RuntimeError('Abra este notebook no Google Colab para carregar o dataset do Google Drive.')
from google.colab import drive
# conecta o Google Drive de quem executa (pede autorização na 1ª vez)
drive.mount('/content/drive')

# pasta do projeto no Drive e onde ficará a cópia local, mais rápida de ler
DRIVE_BASE = Path('/content/drive/MyDrive/Fase 6 - Dataset')
DRIVE_DATASET = DRIVE_BASE / 'dataset_fase6'
DRIVE_ZIP = DRIVE_BASE / 'dataset_fase6.zip'
LOCAL_DATASET = Path('/content/farmtech/data/dataset_fase6')
# cria a pasta local de trabalho, se ainda não existir
LOCAL_DATASET.parent.mkdir(parents=True, exist_ok=True)

# 1ª opção: dataset já descompactado no Drive -> copia para o disco local
if (DRIVE_DATASET / 'data_info.json').is_file():
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET, dirs_exist_ok=True)
# 2ª opção: só o ZIP no Drive -> descompacta no disco local
elif DRIVE_ZIP.is_file():
    with zipfile.ZipFile(DRIVE_ZIP) as z:
        # segurança: recusa arquivos do ZIP com caminho absoluto ou '..'
        for item in z.infolist():
            relativo = Path(item.filename)
            if relativo.is_absolute() or '..' in relativo.parts:
                raise ValueError('Caminho inválido encontrado no ZIP do dataset.')
        # extrai imagens, rótulos e metadados
        z.extractall(LOCAL_DATASET)
    # Aceita tanto ZIP com a pasta dataset_fase6/ na raiz quanto ZIP com os arquivos na raiz.
    nested = LOCAL_DATASET / 'dataset_fase6'
    if (nested / 'data_info.json').is_file():
        for item in nested.iterdir():
            destino = LOCAL_DATASET / item.name
            if item.is_dir(): shutil.copytree(item, destino, dirs_exist_ok=True)
            else: shutil.copy2(item, destino)
        shutil.rmtree(nested)
else:
    # nenhuma das opções: avisa exatamente o que enviar ao Drive
    raise FileNotFoundError(
        'Dataset não encontrado no Drive. Crie a pasta "Fase 6 - Dataset" em Meu Drive '
        'e envie para ela o arquivo data/Fase 6 - Dataset/dataset_fase6.zip do repositório, sem renomeá-lo.'
    )

# confere se a extração produziu o arquivo de metadados esperado
if not (LOCAL_DATASET / 'data_info.json').is_file():
    raise FileNotFoundError(f'Não encontrei data_info.json em {LOCAL_DATASET}. Confira o ZIP enviado.')
print('Dataset carregado do Google Drive:', DRIVE_DATASET if DRIVE_DATASET.exists() else DRIVE_ZIP)
print('Cópia temporária para processamento:', LOCAL_DATASET)

In [ ]:
import os, sys, json, shutil
from pathlib import Path
from datetime import datetime, timezone

# Os dados foram copiados do Drive para o disco temporário do Colab na célula anterior.
IN_COLAB = 'google.colab' in sys.modules
# pasta raiz de trabalho: no Colab fica em /content/farmtech
RAIZ = Path('/content/farmtech') if IN_COLAB else (Path.cwd().resolve().parent if Path.cwd().name == 'src' else Path.cwd().resolve())
RAIZ.mkdir(parents=True, exist_ok=True)
# caminhos usados no resto do notebook
WORK = str(RAIZ)
DATASET = str(RAIZ/'data'/'dataset_fase6')
# garante que a célula do Drive foi executada antes
if not (Path(DATASET)/'data_info.json').is_file():
    raise FileNotFoundError('Dataset ausente. Execute primeiro a célula de conexão com o Google Drive.')

# Resultados de execução são gravados no Drive quando disponível.
pasta_resultados = RAIZ/'assets'/'execucoes'
if IN_COLAB:
    pasta_resultados = Path('/content/drive/MyDrive/Fase 6 - Dataset/resultados')
# cada execução ganha uma pasta própria com data e hora (UTC), sem sobrescrever as anteriores
RUNS = str(pasta_resultados/datetime.now(timezone.utc).strftime('execucao_%Y%m%dT%H%M%S_%fZ'))
Path(RUNS).mkdir(parents=True, exist_ok=False)
# metadados do dataset: nomes das classes e equivalentes no COCO
INFO = json.loads((Path(DATASET)/'data_info.json').read_text(encoding='utf-8'))
CLASSES, COCO_EQUIV = INFO['classes'], INFO['coco_equiv']
print('Classes:', CLASSES, '\nDataset:', DATASET, '\nResultados:', RUNS)

In [ ]:
%matplotlib inline
# ============ PARÂMETROS DOS EXPERIMENTOS ============
EPOCAS = [30, 60]       # as duas simulações pedidas no enunciado (bem diferentes entre si)
IMG_SIZE = 320          # mesma resolução nas duas rodadas; viabiliza CPU
BATCH = 16              # imagens processadas por vez em cada passo do treino
CONF = 0.25             # confiança mínima para considerar uma detecção
SEMENTE = 42            # reprodutibilidade dos experimentos

# Bibliotecas de apoio e limite de threads de CPU (deixa os tempos mais estáveis)
import time, random, subprocess
import torch
# usa no máximo 4 núcleos de CPU em PyTorch, OpenMP, MKL e TensorFlow
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS)
os.environ['TF_NUM_INTRAOP_THREADS'] = str(CPU_THREADS)
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
# bibliotecas de dados, gráficos e imagens
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from IPython.display import display, Markdown

# semente fixa: os sorteios se repetem a cada execução
random.seed(SEMENTE); np.random.seed(SEMENTE)
COR_30, COR_60 = '#2a78d6', '#eb6834'                            # cores fixas das duas simulações
CORES_MODELOS = ['#2a78d6', '#eb6834', '#1baf7a']                # YOLO custom, YOLO padrão, CNN
# função que padroniza o visual de todos os gráficos
def estilo_graficos():
    # estilo limpo e consistente em todos os gráficos (o YOLOv5 troca o estilo do matplotlib ao ser importado)
    plt.style.use('default')
    plt.rcParams.update({'axes.grid': True, 'grid.alpha': .25, 'axes.spines.top': False,
                         'axes.spines.right': False, 'lines.linewidth': 2, 'figure.facecolor': 'white'})
estilo_graficos()

# A aceleração precisa estar disponível ao PyTorch, não só ao driver.
# semente também no PyTorch
torch.manual_seed(SEMENTE)
# usa a GPU (dispositivo '0') quando o PyTorch enxerga CUDA; senão, CPU
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
if DEVICE == '0':
    torch.cuda.manual_seed_all(SEMENTE)
    # desliga a autoescolha de algoritmos da GPU para resultados mais repetíveis
    torch.backends.cudnn.benchmark = False
    gpu = torch.cuda.get_device_name(0)
else:
    gpu = ''
    # sem GPU, lote menor para caber na memória RAM
    BATCH = 8
print('GPU CUDA:', gpu or 'indisponível; CPU selecionada')
print('Batch:', BATCH, '| dispositivo:', DEVICE)
# MPS não é ativado automaticamente: o protocolo usa CUDA ou CPU para YOLO e TensorFlow.

### 1.1 Conferência do dataset
Antes de treinar, verificamos se cada etapa tem a quantidade esperada de imagens e se **toda imagem tem seu rótulo**. Um rótulo faltando é o erro mais comum e derruba a qualidade do modelo sem dar nenhum aviso.

| Etapa | Imagens | Papel |
|---|---|---|
| `train` | 64 | O modelo **aprende** com elas |
| `val` | 8 | Acompanham o treino e escolhem o melhor modelo |
| `test` | 8 | **Nunca** são vistas no treino: medem o desempenho real |

In [ ]:
import glob
import hashlib
from collections import Counter
# extensões de imagem aceitas
EXT = ('.jpg', '.jpeg', '.png')
# lista, em ordem alfabética, as imagens de uma etapa (train, val ou test)
def listar_imagens(split):
    return sorted(str(p) for p in (Path(DATASET)/'images'/split).iterdir() if p.suffix.lower() in EXT)
# caminho do .txt YOLO que acompanha cada imagem
def rotulo_de(img_path):
    p = Path(img_path)
    return str(Path(DATASET)/'labels'/p.parent.name/(p.stem+'.txt'))
# lê e valida o rótulo: 5 valores por linha, classe válida e caixa dentro da foto
def ler_rotulos(img_path):
    p = Path(rotulo_de(img_path))
    if not p.is_file(): raise ValueError(f'Rótulo ausente: {p}')
    # cada linha do .txt: classe, x_centro, y_centro, largura, altura (normalizados 0–1)
    linhas = [l.split() for l in p.read_text().splitlines() if l.strip()]
    if not linhas or any(len(l) != 5 for l in linhas): raise ValueError(f'Rótulo inválido: {p}')
    dados = np.array(linhas, dtype=float)
    # confere cada caixa: números finitos, classe inteira e caixa dentro dos limites da imagem
    for c, x, y, w, h in dados:
        if not (np.isfinite([c,x,y,w,h]).all() and c == int(c) and 0 <= c < len(CLASSES)
                and 0 < w <= 1 and 0 < h <= 1 and x-w/2 >= -1e-5 and x+w/2 <= 1.00001
                and y-h/2 >= -1e-5 and y+h/2 <= 1.00001):
            raise ValueError(f'Classe/coordenadas inválidas: {p}')
    # o protocolo de classificação da Entrega 2 assume um único objeto por foto
    if len(dados) != 1: raise ValueError(f'O protocolo de classificação exige um objeto por foto: {p}')
    return dados
# classe real da imagem (gabarito), lida do próprio rótulo
def classe_verdadeira(img_path):
    return int(ler_rotulos(img_path)[0,0])
# Auditoria do dataset: contagens, rótulos órfãos e imagens repetidas
linhas, hashes, manifesto = [], {}, []
# cada etapa deve ter 32/4/4 imagens por classe, como pede o enunciado
for split, esperado in [('train',32), ('val',4), ('test',4)]:
    imgs = listar_imagens(split)
    # conta quantas imagens de cada classe existem nesta etapa
    contagem = Counter(classe_verdadeira(p) for p in imgs)
    assert contagem == Counter({0:esperado, 1:esperado}), f'Divisão incorreta: {split}: {contagem}'
    # toda imagem precisa de um rótulo e todo rótulo de uma imagem
    esperados = {Path(p).stem for p in imgs}
    presentes = {p.stem for p in (Path(DATASET)/'labels'/split).glob('*.txt') if p.name != 'classes.txt'}
    assert esperados == presentes, f'Rótulos ausentes ou órfãos em {split}'
    for p in imgs:
        with Image.open(p) as im:
            rgb = im.convert('RGB')
            # impressão digital dos pixels: detecta a mesma foto repetida em etapas diferentes
            digest = hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()
        assert digest not in hashes, f'Imagem repetida: {p} e {hashes.get(digest)}'
        hashes[digest] = p
        # registra imagem, etapa, classe e hashes para auditoria posterior
        manifesto.append({'imagem':str(Path(p).relative_to(DATASET)), 'split':split,
                          'classe':classe_verdadeira(p), 'sha256_pixels':digest,
                          'sha256_rotulo':hashlib.sha256(Path(rotulo_de(p)).read_bytes()).hexdigest()})
    linhas.append({'etapa':split, 'imagens':len(imgs), **{c:contagem[i] for i,c in enumerate(CLASSES)}})
# tabela-resumo das contagens por etapa e classe
tabela_dataset = pd.DataFrame(linhas).set_index('etapa')
display(tabela_dataset)
# salva o manifesto junto com os resultados desta execução
pd.DataFrame(manifesto).to_csv(Path(RUNS)/'manifesto_dataset.csv', index=False)
print('80 imagens válidas; 32/4/4 por classe; caixas válidas; sem duplicatas exatas de pixels.')
print('Fotos quase iguais exigem revisão visual; hashes não detectam vazamento por mesma sessão fotográfica.')

In [ ]:
# Amostra visual do dataset rotulado: 4 imagens de cada classe com a caixa desenhada no Make Sense
# pega as 4 primeiras imagens de treino de cada classe
amostra = []
for c in range(len(CLASSES)):
    amostra += [p for p in listar_imagens('train') if classe_verdadeira(p) == c][:4]

# grade 2 × 4: uma linha por classe
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.flat, amostra):
    # abre a foto e guarda a largura e a altura em pixels
    img = Image.open(path); w, h = img.size
    ax.imshow(img); ax.axis('off'); ax.set_title(os.path.basename(path), fontsize=9)
    for linha in open(rotulo_de(path)):                         # cada linha = um objeto rotulado
        c, xc, yc, bw, bh = map(float, linha.split())           # formato YOLO: centro e tamanho (0–1)
        x0, y0 = (xc - bw / 2) * w, (yc - bh / 2) * h           # converte para pixels (canto superior esquerdo)
        # desenha a caixa e escreve o nome da classe acima dela
        ax.add_patch(patches.Rectangle((x0, y0), bw * w, bh * h, fill=False, color='lime', lw=2.5))
        ax.text(x0, y0 - 10, CLASSES[int(c)], color='black', fontsize=10, weight='bold',
                bbox=dict(facecolor='lime', edgecolor='none', pad=2))
plt.suptitle('Amostra do dataset de treino com os rótulos do Make Sense', fontsize=14)
plt.tight_layout(); plt.show()

### 1.1.1 Evidências da rotulação (Make Sense AI)
As caixas foram desenhadas e revisadas no [Make Sense AI](https://www.makesense.ai/) e exportadas no formato YOLO (um `.txt` por imagem; classes `0 = xicara` e `1 = tesoura`). A rotulação foi feita sobre as fotos originais do celular; cada rótulo foi ligado à sua foto pela comparação exata dos arquivos e renomeado conforme o `mapa_nomes.csv`. As 80 caixas foram conferidas visualmente antes do treino.

**1. As 80 fotos importadas no Make Sense**

![Projeto e importação](https://raw.githubusercontent.com/jonathasArauj/FIAP_IA_TURMAB_GRUPO_BH_MG2/main/ANO1/FASE%206/assets/makesense/01_projeto_importacao.png)

**2. Caixas revisadas: a tesoura inteira (das argolas à ponta das lâminas) e a xícara com a alça dentro da caixa**

![Caixas revisadas](https://raw.githubusercontent.com/jonathasArauj/FIAP_IA_TURMAB_GRUPO_BH_MG2/main/ANO1/FASE%206/assets/makesense/02_caixas_revisadas.png)

**3. Exportação das anotações no formato YOLO**

![Exportação YOLO](https://raw.githubusercontent.com/jonathasArauj/FIAP_IA_TURMAB_GRUPO_BH_MG2/main/ANO1/FASE%206/assets/makesense/03_exportacao_yolo.png)

**4. Conferência final das 80 caixas** (prancha gerada a partir dos rótulos usados no treino)

![Conferência de todas as caixas](https://raw.githubusercontent.com/jonathasArauj/FIAP_IA_TURMAB_GRUPO_BH_MG2/main/ANO1/FASE%206/assets/makesense/04_conferencia_todas_as_caixas.jpg)


### 1.2 Análise exploratória do dataset
Antes de treinar, olhamos **como os objetos aparecem** nas fotos. Objetos muito pequenos, sempre no centro ou sempre na mesma orientação limitam o que o modelo aprende e explicam erros mais tarde.

In [ ]:
# Análise exploratória só em treino e validação (o teste fica guardado)
registros = []
for split in ['train', 'val']:  # não explorar o teste antes de fixar o protocolo
    for p in listar_imagens(split):
        # tamanho da foto, usado para saber se é retrato ou paisagem
        w, h = Image.open(p).size
        for linha in open(rotulo_de(p)):
            c, xc, yc, bw, bh = map(float, linha.split())
            # guarda classe, área relativa da caixa, posição do centro e orientação
            registros.append({'etapa': split, 'classe': CLASSES[int(c)], 'area_%': 100 * bw * bh, 'xc': xc, 'yc': yc,
                              'orientação': 'retrato' if h > w else 'paisagem'})
# tabela com uma linha por objeto rotulado
eda = pd.DataFrame(registros)

# três gráficos: tamanho do objeto, posição do centro e orientação da foto
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(17, 4.8))
for cor, (classe, g) in zip([COR_30, COR_60], eda.groupby('classe')):
    a1.hist(g['area_%'], bins=12, alpha=.75, color=cor, label=classe, edgecolor='white')
    a2.scatter(g['xc'], g['yc'], s=45, alpha=.8, color=cor, label=classe, edgecolor='white')
a1.set_title('Tamanho do objeto (% da área da foto)'); a1.set_xlabel('% da foto ocupada pela caixa'); a1.legend(frameon=False)
a2.set_title('Posição do centro da caixa'); a2.set_xlim(0, 1); a2.set_ylim(1, 0); a2.set_aspect('equal')
a2.set_xlabel('horizontal (0 = esquerda)'); a2.set_ylabel('vertical (0 = topo)'); a2.legend(frameon=False, loc='lower right')
# linhas pontilhadas que dividem a foto em 9 regiões
for v in (1 / 3, 2 / 3):
    a2.axvline(v, color='gray', lw=.8, ls=':'); a2.axhline(v, color='gray', lw=.8, ls=':')
# quantas fotos de cada classe estão em retrato e em paisagem
orient = eda.groupby(['classe', 'orientação']).size().unstack(fill_value=0)
orient.plot.bar(ax=a3, color=['#1baf7a', '#6250d6'][:orient.shape[1]], rot=0, edgecolor='white', width=.6)
a3.set_title('Orientação das fotos'); a3.set_xlabel(''); a3.legend(frameon=False); a3.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

# fração de objetos no quadro central e fração de objetos muito pequenos (< 5% da foto)
centrais = (eda.xc.between(1 / 3, 2 / 3) & eda.yc.between(1 / 3, 2 / 3)).mean()
pequenos = (eda['area_%'] < 5).mean()
# texto gerado a partir dos números acima
display(Markdown('\n'.join([
    f"- Os objetos ocupam em média **{eda['area_%'].mean():.0f}%** da foto (de {eda['area_%'].min():.0f}% a {eda['area_%'].max():.0f}%).",
    f"- **{centrais:.0%}** dos objetos estão no quadro central (linhas pontilhadas). "
    + ('Concentração alta: o modelo pode estranhar objetos perto das bordas.' if centrais > 0.7 else 'Há boa variação de posição.'),
    f"- **{pequenos:.0%}** das caixas ocupam menos de 5% da foto. "
    + ('Objetos pequenos são os mais difíceis de detectar.' if pequenos > 0.2 else 'Poucos objetos muito pequenos.')])))

## 2. Entrega 1 — YOLO customizado
### 2.1 Configuração
Usamos **YOLOv5s pré-treinado no COCO** e ajustamos seus pesos para xícara e tesoura. A revisão do código é fixada para permitir repetir o experimento. O arquivo YAML informa classes e caminhos de treino, validação e teste.

Cada execução gera uma pasta própria em `MyDrive/Fase 6 - Dataset/resultados/execucao_<data>/` (Colab com Drive), com pesos, métricas, gráficos e imagens detectadas.


In [ ]:
import subprocess
# fixa a versão exata do YOLOv5 para a execução poder ser repetida
YOLO_REV = 'f9578796b8a64889aa9e66dc06914d19f40d338e'  # versão do YOLOv5 utilizada no experimento
YOLO_DIR = str(Path(WORK)/'.dependencias'/'yolov5')
Path(YOLO_DIR).parent.mkdir(parents=True, exist_ok=True)
# baixa o repositório oficial do YOLOv5 só na primeira vez
if not Path(YOLO_DIR).exists():
    subprocess.run(['git', 'clone', 'https://github.com/ultralytics/yolov5.git', YOLO_DIR], check=True)
# muda para a versão fixada
subprocess.run(['git', '-C', YOLO_DIR, 'checkout', '--detach', YOLO_REV], check=True)
YOLO_COMMIT = subprocess.check_output(['git', '-C', YOLO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
# instala as dependências do YOLOv5 no mesmo Python do notebook
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(Path(YOLO_DIR)/'requirements.txt'), 'opencv-python==4.14.0.94'], check=True)
# permite importar val.py do YOLOv5 como biblioteca
if YOLO_DIR not in sys.path:
    sys.path.insert(0, YOLO_DIR)
print('YOLOv5:', YOLO_COMMIT)

In [ ]:
import yaml, torch

# Arquivo de configuração do dataset no formato que o YOLOv5 espera
data_cfg = {
    'path': DATASET,            # raiz do dataset
    'train': 'images/train',    # 64 imagens para aprender
    'val': 'images/val',        # 8 imagens para acompanhar o treino
    'test': 'images/test',      # 8 imagens reservadas para o teste final
    'nc': len(CLASSES),         # número de classes
    'names': CLASSES,           # nomes das classes, na ordem dos rótulos
}
# grava o YAML dentro da pasta do YOLOv5 e mostra o conteúdo
DATA_YAML = f'{YOLO_DIR}/data_fase6.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f, allow_unicode=True)
print(open(DATA_YAML).read())
DEVICE = '0' if torch.cuda.is_available() else 'cpu'                     # usa a GPU quando existir
print('Dispositivo de treino:', torch.cuda.get_device_name(0) if DEVICE == '0' else 'CPU')

### 2.2 Treinamento — 30 × 60 épocas
- **Época** = uma passada completa do modelo por todas as imagens de treino.
- **Transfer learning:** os dois treinos partem do `yolov5s.pt`, o YOLOv5 *small* já treinado no dataset COCO (118 mil imagens). A rede já sabe enxergar bordas, texturas e formas; nós só a ensinamos a reconhecer **os nossos dois objetos**. É isso que torna viável treinar com só 64 imagens.
- **Parâmetro experimental:** orçamento de 30 ou 60 épocas; imagem, batch, semente e pesos iniciais são mantidos. O agendamento da taxa de aprendizado depende do total de épocas; portanto, 60 épocas não equivalem simplesmente a continuar o treino de 30. São dois treinamentos independentes.

**Hipótese:** mais épocas tendem a melhorar as métricas, mas com um dataset pequeno existe o risco de **overfitting** (o modelo decora o treino e piora em imagens novas), e o custo de treino aumenta. Uma única semente não permite atribuir pequenas diferenças exclusivamente ao orçamento.

**Resolução:** 320 × 320 nas duas rodadas, para execução viável em CPU. As fotos originais são preservadas. O enunciado não fixa resolução; a resolução é igual nas duas rodadas.

In [ ]:
# Treina o YOLOv5s duas vezes, mudando só o número de épocas (30 e 60)
tempos_treino = {}
for ep in EPOCAS:
    destino = Path(RUNS)/f'treino_{ep}ep'
    # não sobrescreve um treino já feito nesta pasta de execução
    if destino.exists():
        raise FileExistsError('Inicie uma nova execução para treinar novamente.')
    # mesma imagem, lote, semente e pesos iniciais (yolov5s.pt do COCO) nas duas rodadas
    comando = [sys.executable, str(Path(YOLO_DIR)/'train.py'), '--img',str(IMG_SIZE),
               '--batch-size',str(BATCH),'--epochs',str(ep),'--data',DATA_YAML,
               '--weights',str(Path(YOLO_DIR)/'yolov5s.pt'),'--device',DEVICE,
               '--workers','0','--seed',str(SEMENTE),'--patience','0',
               '--project',RUNS,'--name',f'treino_{ep}ep']
    # cronometra o treino completo
    inicio = time.perf_counter()
    # roda o train.py e guarda toda a saída num arquivo de log
    with open(Path(RUNS)/f'treino_{ep}ep.log','w') as log:
        subprocess.run(comando, cwd=YOLO_DIR, env={**os.environ,'MPLBACKEND':'Agg','WANDB_MODE':'disabled'}, stdout=log, stderr=subprocess.STDOUT, check=True)
    # tempo de treino em minutos
    tempos_treino[ep] = (time.perf_counter()-inicio)/60
    # confere se o treino rodou todas as épocas e salvou o best.pt
    df = pd.read_csv(destino/'results.csv')
    assert len(df) == ep, f'Treino incompleto: {len(df)}/{ep} épocas'
    assert (destino/'weights/best.pt').is_file()
    print(f'{ep} épocas concluídas em {tempos_treino[ep]:.1f} min')

30 épocas concluídas em 2.4 min


### 2.3 Curvas de treinamento
Lemos o `results.csv` de cada treino. Como ler os gráficos:
- **Perdas (loss):** o erro do modelo, quanto menor melhor. Se a perda de **treino** continua caindo enquanto a de **validação** sobe, é sinal de overfitting.
- **Precisão:** das detecções feitas, quantas estavam certas.
- **Recall:** dos objetos que existiam, quantos foram encontrados.
- **mAP@0.5:** métrica principal de detecção, que exige acertar a classe **e** a posição da caixa (sobreposição ≥ 50%). O **mAP@0.5:0.95** é a versão mais rigorosa, que pede caixas cada vez mais exatas.

In [ ]:
# Lê o histórico de cada treino (uma linha por época)
historico = {}
for ep in EPOCAS:
    df = pd.read_csv(f'{RUNS}/treino_{ep}ep/results.csv')
    df.columns = df.columns.str.strip()                                  # o YOLO grava os nomes com espaços
    historico[ep] = df

# métricas exibidas: perdas de treino e validação, precisão, recall e mAP
graficos = [('train/box_loss', 'Perda da caixa — treino'), ('val/box_loss', 'Perda da caixa — validação'),
            ('val/cls_loss', 'Perda da classe — validação'), ('metrics/precision', 'Precisão (validação)'),
            ('metrics/recall', 'Recall (validação)'), ('metrics/mAP_0.5', 'mAP@0.5 (validação)')]
# um gráfico por métrica, com as duas rodadas sobrepostas
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, (col, titulo) in zip(axes.flat, graficos):
    for ep, cor in zip(EPOCAS, [COR_30, COR_60]):
        ax.plot(historico[ep]['epoch'] + 1, historico[ep][col], color=cor, label=f'{ep} épocas')
    ax.set_title(titulo, fontsize=12); ax.set_xlabel('época')
    ax.legend(frameon=False)
plt.suptitle(f'Evolução do treinamento: {EPOCAS[0]} × {EPOCAS[1]} épocas', fontsize=15); plt.tight_layout(); plt.show()

# tabela-resumo: melhores métricas, perdas finais e tempo de cada treino
resumo_treino = pd.DataFrame({
    f'{ep} épocas': {
        'Melhor mAP@0.5 (val)': df['metrics/mAP_0.5'].max(),
        'Melhor mAP@0.5:0.95 (val)': df['metrics/mAP_0.5:0.95'].max(),
        'Época de maior mAP@0.5:0.95 (não necessariamente best.pt)': int(df['metrics/mAP_0.5:0.95'].idxmax()) + 1,
        'Perda da caixa final (treino)': df['train/box_loss'].iloc[-1],
        'Perda da caixa final (validação)': df['val/box_loss'].iloc[-1],
        'Tempo de treino (min)': tempos_treino.get(ep, np.nan),
    } for ep, df in historico.items()})
display(resumo_treino.round(3))

### 2.4 Validação, seleção e teste
Primeiro avaliamos os dois `best.pt` na **validação**. Escolhemos o orçamento pelo maior mAP@0.5:0.95 nesse conjunto; em empate exato, preferimos 30 épocas pelo menor custo. A decisão é registrada antes de abrir o teste.

Depois avaliamos os dois no teste para cumprir a comparação exigida, sem mudar a escolha, o limiar ou o treinamento em função desses resultados. `best.pt` é escolhido durante cada treino pela função de fitness da implementação do YOLOv5; a seleção entre os dois orçamentos usa a regra explícita acima.

**Métricas diferentes:** mAP integra a curva precisão–recall com diferentes limiares; a avaliação usa `conf_thres=0.001`. As imagens demonstrativas e a classificação usam `CONF=0.25`. Confiança da caixa não é acurácia nem probabilidade calibrada de acerto.


In [ ]:
import val as yolo_val
# Avalia os dois modelos na validação e, depois, no teste
avaliacao = []
for etapa in ['val','test']:
    # antes de abrir o teste, escolhe o modelo pela validação e registra a escolha
    if etapa == 'test':
        validacao = pd.DataFrame(avaliacao)
        # critério: maior mAP@0.5:0.95 na validação
        melhor_ep = max(EPOCAS, key=lambda e: float(validacao.loc[validacao['treino']==f'{e} épocas','mAP@0.5:0.95'].iloc[0]))
        # salva a regra de escolha: prova de que o teste não influenciou a decisão
        Path(RUNS,'selecao_validacao.json').write_text(json.dumps({
            'epocas':melhor_ep,'criterio':'maior mAP@0.5:0.95 na validação; empate: menor orçamento',
            'conf_classificacao':CONF,'teste_usado_na_selecao':False},ensure_ascii=False,indent=2))
        print('Seleção fixada antes do teste:', melhor_ep)
    for ep in EPOCAS:
        # val.py do YOLOv5: devolve precisão, recall, mAP e tempo por imagem
        (p,r,map50,map5095,*_), _, velocidade = yolo_val.run(
            data=DATA_YAML, weights=f'{RUNS}/treino_{ep}ep/weights/best.pt', imgsz=IMG_SIZE,
            task=etapa, device=DEVICE, batch_size=8, workers=0, conf_thres=0.001, iou_thres=0.6,
            project=RUNS,name=f'{etapa}_{ep}ep',exist_ok=True,plots=True,verbose=True)
        # guarda as métricas de cada modelo em cada etapa
        avaliacao.append({'treino':f'{ep} épocas','etapa':'validação' if etapa=='val' else 'teste',
                          'precisão':p,'recall':r,'mAP@0.5':map50,'mAP@0.5:0.95':map5095,
                          'inferência (ms/img)':velocidade[1]})
# tabela final da Entrega 1, também salva em CSV
tabela_avaliacao = pd.DataFrame(avaliacao).set_index(['treino','etapa'])
display(tabela_avaliacao.round(3))
tabela_avaliacao.to_csv(Path(RUNS)/'avaliacao_deteccao.csv')

In [ ]:
# Exibe os gráficos no notebook.
%matplotlib inline
estilo_graficos()
# Matriz de confusão e curva precisão × recall no TESTE, lado a lado para os dois treinos
# linhas = 30 e 60 épocas; colunas = matriz de confusão e curva precisão × recall
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for linha, ep in enumerate(EPOCAS):
    for col, arquivo in enumerate(['confusion_matrix.png', 'PR_curve.png']):
        caminho = f'{RUNS}/test_{ep}ep/{arquivo}'
        axes[linha, col].axis('off')
        # mostra a figura gerada pelo val.py, se existir
        if os.path.exists(caminho):
            axes[linha, col].imshow(Image.open(caminho))
        axes[linha, col].set_title(f'{ep} épocas · {"matriz de confusão" if col == 0 else "curva precisão × recall"} (teste)')
plt.tight_layout(); plt.show()

### 2.5 Imagens de teste processadas
Rodamos o `detect.py` nas 8 imagens de teste. Cada imagem sai com a **caixa**, o **nome da classe** e a **confiança** (0 a 1) da detecção.

In [ ]:
# Exibe os gráficos no notebook.
%matplotlib inline
estilo_graficos()
# Mostra em grade as imagens de teste com as caixas previstas pelo modelo
def mostrar_deteccoes(pasta, titulo):
    # exibe em grade as imagens já processadas (com caixas desenhadas pelo YOLO)
    imgs = sorted(f for f in glob.glob(f'{pasta}/*') if f.lower().endswith(EXT))
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for ax in axes.flat: ax.axis('off')
    for ax, p in zip(axes.flat, imgs):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=9)
    plt.suptitle(titulo, fontsize=14); plt.tight_layout(); plt.show()

# roda o detect.py com o best.pt de cada treino nas 8 imagens de teste
for ep in EPOCAS:
    pesos = f'{RUNS}/treino_{ep}ep/weights/best.pt'
    subprocess.run([sys.executable, str(Path(YOLO_DIR)/'detect.py'), '--weights', pesos, '--source', str(Path(DATASET)/'images/test'), '--img', str(IMG_SIZE), '--conf-thres', str(CONF), '--device', DEVICE, '--project', RUNS, '--name', f'detect_{ep}ep', '--exist-ok'], cwd=YOLO_DIR, env={**os.environ, 'MPLBACKEND':'Agg'}, check=True)
    # exibe as detecções: são os prints para o cliente
    mostrar_deteccoes(f'{RUNS}/detect_{ep}ep', f'Teste — modelo treinado com {ep} épocas')

### 2.6 Resultados da Entrega 1
A célula abaixo gera um **resumo automático** a partir dos números reais desta execução. Logo depois, a análise do grupo.

In [ ]:
# Gera automaticamente o resumo da Entrega 1 a partir da tabela de avaliação
t = tabela_avaliacao
texto = []
# uma frase por treino, com validação, teste e tempo
for ep in EPOCAS:
    va = t.loc[(f'{ep} épocas','validação')]
    te = t.loc[(f'{ep} épocas','teste')]
    texto.append(f"**{ep} épocas:** mAP@0.5:0.95 de validação {va['mAP@0.5:0.95']:.3f}; teste {te['mAP@0.5:0.95']:.3f}; treino {tempos_treino[ep]:.1f} min.")
# frases fixas sobre a escolha do modelo e os limites da conclusão
texto += [f'**Escolha pela validação:** {melhor_ep} épocas. A comparação de teste não altera a seleção.',
          '**Interpretação:** diferença entre duas execuções não prova superioridade estatística nem overfitting. Avaliar conjuntamente perdas de treino/validação, diversidade dos dados e múltiplas sementes.',
          '**Limite:** 8 fotos de teste não demonstram desempenho em produção, novos ambientes ou vídeo.']
display(Markdown('\n\n'.join('- '+x for x in texto)))

### 2.7 Interpretação para o cliente
A seleção é feita pela validação; o teste documenta o comportamento final. Mais épocas podem melhorar a localização, mas um ganho pequeno deve ser ponderado pelo custo e pela variação amostral. mAP@0.5 alto não implica caixas perfeitas: compare também mAP@0.5:0.95 e as imagens anotadas.

A diferença de velocidade entre `val.py`, `detect.py` e a comparação da Entrega 2 não pode ser atribuída somente ao batch sem investigação: os caminhos de execução, aquecimento e etapas cronometradas diferem. A comparação da Entrega 2 usa um protocolo único de leitura da imagem até a decisão, com repetições.

Para a FarmTech, o detector é uma prova de conceito para xícara e tesoura. Próximos passos: coletar novos ambientes e sessões de fotos, incluir múltiplos objetos e oclusões, anotar manualmente as caixas e reservar um teste externo. Fotos da mesma sequência devem permanecer no mesmo split para reduzir vazamento visual.


In [ ]:
# Conclusão autoral da Entrega 1, gerada a partir dos números DESTA execução (nenhum valor é digitado à mão)
e1, e2 = EPOCAS                                       # os dois orçamentos de épocas comparados
br = lambda v, c=3: f'{v:.{c}f}'.replace('.', ',')    # número no padrão brasileiro (vírgula decimal)
def met(ep, etapa, coluna):
    # valor da tabela de avaliação, com as mesmas 3 casas mostradas nas tabelas
    return round(float(tabela_avaliacao.loc[(f'{ep} épocas', etapa), coluna]), 3)

te1, te2 = met(e1, 'teste', 'mAP@0.5:0.95'), met(e2, 'teste', 'mAP@0.5:0.95')
va1, va2 = met(e1, 'validação', 'mAP@0.5:0.95'), met(e2, 'validação', 'mAP@0.5:0.95')
p1, p2 = met(e1, 'teste', 'precisão'), met(e2, 'teste', 'precisão')
r1, r2 = met(e1, 'teste', 'recall'), met(e2, 'teste', 'recall')
a1, a2 = met(e1, 'teste', 'mAP@0.5'), met(e2, 'teste', 'mAP@0.5')
t1, t2 = round(tempos_treino[e1], 3), round(tempos_treino[e2], 3)
ganho = round(te2 - te1, 3)                           # variação absoluta do mAP@0.5:0.95 no teste
ganho_rel = ganho / te1 if te1 else float('nan')      # variação relativa
custo_rel = t2 / t1 - 1                               # aumento relativo do tempo de treino
melhor_epoca = {ep: int(historico[ep]['metrics/mAP_0.5:0.95'].idxmax()) + 1 for ep in EPOCAS}
perda_val = {ep: round(float(historico[ep]['val/box_loss'].iloc[-1]), 3) for ep in EPOCAS}

f = []
# 1) efeito das épocas na localização das caixas (teste)
if ganho != 0:
    verbo = 'elevou' if ganho > 0 else 'reduziu'
    f.append(f'O treino de {e2} épocas {verbo} o mAP@0.5:0.95 no conjunto de teste de {br(te1)} para {br(te2)}: '
             f'variação absoluta de {br(abs(ganho))} (cerca de {br(abs(100 * ganho_rel), 1)}% em relação às {e1} épocas).')
else:
    f.append(f'O mAP@0.5:0.95 no teste ficou em {br(te1)} nos dois treinos.')
# 2) validação e seleção
f.append(f'Na validação, que decidiu a seleção, o valor foi de {br(va1)} para {br(va2)}, e a escolha recaiu sobre o treino de {melhor_ep} épocas.')
# 3) mAP@0.5 e precisão/recall (teste)
if a1 == a2:
    f.append(f'O mAP@0.5 de teste ficou em {br(a1)} nos dois treinos; a diferença entre eles está na qualidade das caixas (mAP@0.5:0.95).')
else:
    f.append(f'O mAP@0.5 de teste passou de {br(a1)} para {br(a2)}; como o mAP@0.5:0.95 exige caixas mais exatas, ele mostra melhor a diferença de localização.')
rec = f'o recall foi de {br(r1)} nos dois' if r1 == r2 else f'o recall passou de {br(r1)} para {br(r2)}'
f.append(f'No teste, a precisão passou de {br(p1)} para {br(p2)} e {rec}.')
# 4) custo x benefício
cmp_ = 'maior' if ganho_rel > custo_rel else 'menor'
f.append(f'O tempo de treino passou de {br(t1, 1)} para {br(t2, 1)} minutos (cerca de {br(t2 / t1, 1)} vez o custo); '
         f'o ganho relativo de mAP@0.5:0.95 ({br(100 * ganho_rel, 1)}%) foi {cmp_} que o aumento relativo de custo ({br(100 * custo_rel, 1)}%).')
# 5) curvas: o treino curto parou cedo demais?
b1, b2 = melhor_epoca[e1], melhor_epoca[e2]
if b1 >= 0.9 * e1:
    f.append(f'Nas curvas, o melhor mAP@0.5:0.95 de validação do treino de {e1} épocas ocorreu na época {b1} (de {e1}), perto do fim: '
             f'o orçamento curto parece ter limitado o resultado, e não o excesso de treino (no de {e2} épocas, a melhor foi a {b2}ª).')
else:
    f.append(f'Nas curvas, o melhor mAP@0.5:0.95 de validação do treino de {e1} épocas ocorreu na época {b1} (de {e1}), antes do fim, '
             f'o que sugere estabilização (no de {e2} épocas, a melhor foi a {b2}ª).')
if perda_val[e2] < perda_val[e1]:
    f.append(f'A perda de caixa na validação terminou menor com {e2} épocas ({br(perda_val[e2])} contra {br(perda_val[e1])}), sem sinal de piora nesta amostra.')
elif perda_val[e2] > perda_val[e1]:
    f.append(f'A perda de caixa na validação terminou maior com {e2} épocas ({br(perda_val[e2])} contra {br(perda_val[e1])}), possível sinal de overfitting a investigar.')
else:
    f.append(f'A perda de caixa na validação terminou igual nos dois treinos ({br(perda_val[e1])}).')
# 6) ressalva
f.append(f'Ainda assim, a conclusão é específica deste conjunto pequeno ({len(listar_imagens("train"))} imagens de treino, {len(listar_imagens("test"))} de teste, uma única semente): '
         'execuções repetidas variam (GPU, ordem dos lotes), então diferenças pequenas pedem mais sementes e uma amostra externa maior antes de se afirmar que o ganho se generaliza. '
         'A seleção foi feita pela validação, mantendo o teste separado da escolha do modelo.')
display(Markdown(f'**Conclusão autoral — {e1} versus {e2} épocas** (gerada automaticamente a partir desta execução). ' + ' '.join(f)))


---
## 3. Entrega 2 — Comparação de modelos
Na visão computacional não existe solução "melhor" em absoluto: depende do cenário. Por isso colocamos três abordagens na **mesma tarefa** (dizer se cada uma das 8 imagens de teste contém o objeto A ou o B), **nas mesmas imagens** e no **mesmo hardware**.

| Abordagem | O que é | Treino nosso? | Localiza o objeto? |
|---|---|---|---|
| **YOLO customizado** | YOLOv5s ajustado na Entrega 1 | Sim (transfer learning) | Sim |
| **YOLO padrão** | YOLOv3 COCO (YOLO tradicional do material de Redes Neurais), com inferência OpenCV DNN | Não | Sim |
| **CNN do zero** | Rede convolucional criada e treinada por nós, sem pesos prontos | Sim (do zero) | Não, só classifica |

**Critérios:** facilidade de uso/integração · precisão · tempo de treinamento · tempo de inferência.

> **Como as detecções viram classificação:** para os dois YOLO, a classe prevista da imagem é a da detecção com **maior confiança**. Se nenhuma detecção passar do limiar, conta como **erro ("não detectou")**.

### 3.1 YOLO customizado × YOLO padrão
O modelo customizado é o treino com **maior mAP@0.5:0.95 na validação**. Escolher pelo teste seria "espiar a prova": o teste fica reservado só para a medida final.

O YOLO padrão conhece as 80 classes do COCO. Mapeamos cada objeto para o nome equivalente no COCO (`COCO_EQUIV`) e ignoramos as demais classes que ele detectar. Se não restar detecção de `cup` ou `scissors`, registramos abstenção (`-1`), contada como erro. Isso não equivale a classificar formalmente como “vaso”.

**Tempo:** batch 1, três aquecimentos e cinco passagens por imagem; leitura, pré-processamento, rede, pós-processamento e decisão incluídos. O YOLO usa 320 px e a CNN 128 px, portanto comparamos pipelines de uso, não custo isolado das arquiteturas. O cache do sistema operacional também influencia a leitura.

**Correspondência ao material:** usamos configuração e pesos YOLOv3/Darknet, com backend OpenCV DNN. Não reproduzimos o executável C da aula. Arquitetura, backend e pré-processamento diferem do YOLOv5 customizado: não atribuímos toda diferença apenas ao ajuste dos pesos. A inferência comparativa dos três modelos usa CPU.


In [ ]:
# Entrega 2: as três abordagens são avaliadas nas MESMAS 8 imagens de teste
imgs_teste = listar_imagens('test')
y_true = [classe_verdadeira(p) for p in imgs_teste]                      # gabarito das 8 imagens de teste

# melhor_ep foi fixado na etapa de validação, antes de avaliar o teste.
print(f'Modelo customizado escolhido pela validação: {melhor_ep} épocas')
import shutil
# guarda uma cópia do modelo escolhido como best.pt
shutil.copy(f'{RUNS}/treino_{melhor_ep}ep/weights/best.pt', f'{RUNS}/best.pt')   # modelo selecionado
print(f'Cópia salva em {RUNS}/best.pt')

# protocolo de tempo: 5 passagens por imagem, depois de 3 aquecimentos
REPETICOES = 5
latencias = {}
# na GPU, espera o cálculo terminar antes de parar o cronômetro
def sincronizar_torch():
    if DEVICE == '0': torch.cuda.synchronize()
# mede o tempo de cada previsão (do arquivo no disco até a classe) e guarda as previsões
def medir(predict, chave, sync=lambda: None):
    # Mesmo limite: caminho no disco -> decisão de classe; batch=1, aquecimento fora da medida.
    for _ in range(3): predict(imgs_teste[0])
    preds, ms = [], []
    for repeticao in range(REPETICOES):
        for p in imgs_teste:
            sync(); inicio = time.perf_counter()
            pred = predict(p)
            sync(); ms.append((time.perf_counter()-inicio)*1000)
            if repeticao == 0: preds.append(int(pred))
    latencias[chave] = ms
    return preds, float(np.mean(ms))
# transforma as detecções do YOLO numa classe: fica com a de maior confiança
def avaliar_yolo(modelo, mapa_nomes, chave):
    def predict(p):
        with torch.inference_mode():
            det = modelo(p, size=IMG_SIZE).pandas().xyxy[0]
        # ignora detecções de classes que não interessam
        det = det[det['name'].isin(mapa_nomes)]
        return mapa_nomes[det.sort_values('confidence').iloc[-1]['name']] if len(det) else -1
    return medir(predict, chave, sincronizar_torch)

# carrega o YOLO customizado escolhido pela validação (na CPU, para comparar tempos)
yolo_custom = torch.hub.load(YOLO_DIR, 'custom', path=f'{RUNS}/treino_{melhor_ep}ep/weights/best.pt', source='local', verbose=False, device='cpu')
yolo_custom.conf = CONF
# previsões e tempo médio do YOLO customizado
pred_custom, inf_custom = avaliar_yolo(yolo_custom, {c: i for i, c in enumerate(CLASSES)}, 'YOLO customizado')

# Configuração e pesos do YOLOv3/Darknet (YOLO tradicional do material de Redes Neurais).
import cv2, urllib.request, hashlib
# mesmo limite de threads de CPU para o OpenCV
cv2.setNumThreads(CPU_THREADS)
V3_DIR = Path(WORK)/'.dependencias'/'yolov3'; V3_DIR.mkdir(parents=True, exist_ok=True)
# arquivos do YOLOv3 original: arquitetura, nomes das 80 classes do COCO e pesos
recursos_v3 = {
    'yolov3.cfg': 'https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg',
    'coco.names': 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names',
    'yolov3.weights': 'https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov3.weights',
}
# baixa cada arquivo só se ainda não existir
for arquivo, url in recursos_v3.items():
    destino = V3_DIR/arquivo
    if not destino.exists():
        temporario = destino.with_suffix(destino.suffix+'.download')
        urllib.request.urlretrieve(url, temporario)
        temporario.replace(destino)
print('YOLOv3 pesos SHA-256:', hashlib.sha256((V3_DIR/'yolov3.weights').read_bytes()).hexdigest())
coco_nomes = (V3_DIR/'coco.names').read_text().splitlines()
# carrega o YOLOv3 com o módulo DNN do OpenCV, rodando na CPU
rede_v3 = cv2.dnn.readNetFromDarknet(str(V3_DIR/'yolov3.cfg'), str(V3_DIR/'yolov3.weights'))
rede_v3.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
rede_v3.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
# mesma resolução de entrada do YOLO customizado
yolo_padrao = cv2.dnn_DetectionModel(rede_v3)
yolo_padrao.setInputParams(size=(IMG_SIZE,IMG_SIZE), scale=1/255, swapRB=True, crop=False)
# classe do COCO equivalente a cada objeto (cup -> xicara, scissors -> tesoura)
mapa_v3 = {COCO_EQUIV[c]: i for i,c in enumerate(CLASSES)}
# roda o YOLOv3 numa imagem e devolve (classe COCO, confiança, caixa) de cada detecção
def deteccoes_v3(p):
    im = cv2.imread(str(p))
    if im is None: raise ValueError(f'Imagem ilegível: {p}')
    ids, scores, boxes = yolo_padrao.detect(im, confThreshold=CONF, nmsThreshold=0.45)
    return im, [(coco_nomes[int(i)],float(s),box) for i,s,box in zip(np.asarray(ids).reshape(-1),np.asarray(scores).reshape(-1),boxes)]
# fica só com cup/scissors; sem nenhuma delas, a resposta é -1 (abstenção, conta como erro)
def prever_v3(p):
    _, dets = deteccoes_v3(p)
    validas = [d for d in dets if d[0] in mapa_v3]
    return mapa_v3[max(validas,key=lambda d:d[1])[0]] if validas else -1
# previsões e tempo médio do YOLO padrão
pred_padrao, inf_padrao = medir(prever_v3, 'YOLO padrão')

# converte índices em nomes de classe para exibir a tabela
nomes = lambda preds: [CLASSES[i] if i >= 0 else '—' for i in preds]
display(pd.DataFrame({'imagem': [os.path.basename(p) for p in imgs_teste], 'real': nomes(y_true),
                      'YOLO customizado': nomes(pred_custom), 'YOLO padrão': nomes(pred_padrao)}))

In [ ]:
%matplotlib inline
estilo_graficos()
# Desenha as detecções do YOLOv3 nas imagens de teste e salva os prints
pasta_v3 = Path(RUNS)/'detect_padrao'; pasta_v3.mkdir(exist_ok=True)
for p in imgs_teste:
    im, dets = deteccoes_v3(p)
    # uma caixa verde com o nome da classe COCO e a confiança
    for nome,score,(x,y,w,h) in dets:
        cv2.rectangle(im,(x,y),(x+w,y+h),(0,180,0),2)
        cv2.putText(im,f'{nome} {score:.2f}',(x,max(20,y-8)),cv2.FONT_HERSHEY_SIMPLEX,.7,(0,180,0),2)
    # salva a imagem anotada na pasta da execução
    cv2.imwrite(str(pasta_v3/Path(p).name),im)
mostrar_deteccoes(str(pasta_v3), 'YOLOv3 COCO, sem customização (OpenCV DNN)')

### 3.2 CNN treinada do zero
Uma rede convolucional **sem nenhum conhecimento prévio**: todos os pesos começam aleatórios e são aprendidos só com as nossas 64 imagens.

| Camada | Função |
|---|---|
| `Rescaling` | Converte os pixels de 0–255 para 0–1 |
| `RandomFlip` / `RandomRotation` / `RandomZoom` | **Data augmentation**: cria variações a cada época para compensar o dataset pequeno |
| 3 × (`Conv2D` + `MaxPooling2D`) | Extraem características: bordas → formas → partes do objeto |
| `GlobalAveragePooling2D` | Resume cada mapa de características em um número (bem menos parâmetros que `Flatten`) |
| `Dropout(0.3)` | Desliga 30% dos neurônios no treino para reduzir overfitting |
| `Dense(1, sigmoid)` | Saída: probabilidade de a imagem ser do objeto B |

O **early stopping** encerra o treino quando a perda de validação para de melhorar e restaura os melhores pesos.

In [ ]:
# Exibe os gráficos no notebook.
%matplotlib inline
estilo_graficos()
# CNN do zero em TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import layers, models
# usa a GPU no TensorFlow só quando o PyTorch também está usando GPU
TF_GPUS = tf.config.list_physical_devices('GPU')
if DEVICE == 'cpu':
    tf.config.set_visible_devices([], 'GPU')
TF_DEVICE = '/GPU:0' if DEVICE == '0' and TF_GPUS else '/CPU:0'
print('TensorFlow:', TF_DEVICE)
if DEVICE == '0' and not TF_GPUS:
    raise RuntimeError('PyTorch tem CUDA, mas TensorFlow não. Use Colab T4 com ambos disponíveis ou CPU para ambos.')
# memória da GPU sob demanda, para dividir com o PyTorch
for g in TF_GPUS:
    tf.config.experimental.set_memory_growth(g, True)                   # divide a GPU com o PyTorch
# semente também no TensorFlow
tf.keras.utils.set_random_seed(SEMENTE)

TAM = 128                                                                # imagens reduzidas para 128 × 128

def carregar(split):
    # carrega as imagens de um split como matriz numpy (N, 128, 128, 3) e os rótulos (N,)
    paths = listar_imagens(split)
    X = np.stack([np.asarray(Image.open(p).convert('RGB').resize((TAM, TAM))) for p in paths]).astype('float32')
    y = np.array([classe_verdadeira(p) for p in paths])
    return X, y

# carrega treino, validação e teste nas mesmas divisões do YOLO
X_tr, y_tr = carregar('train'); X_va, y_va = carregar('val'); X_te, y_te = carregar('test')
print('treino', X_tr.shape, '| validação', X_va.shape, '| teste', X_te.shape)

# arquitetura: aumento de dados + 3 blocos convolucionais + classificador
def criar_cnn():
    return models.Sequential([
        layers.Input((TAM, TAM, 3)),
        # normaliza os pixels de 0–255 para 0–1
        layers.Rescaling(1 / 255),
        # aumento de dados: espelha, gira e aproxima as fotos (só no treino)
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),                                      # até ±36°
        layers.RandomZoom(0.15),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),  # bloco 1: bordas e texturas
        layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),  # bloco 2: formas
        layers.Conv2D(128, 3, activation='relu'), layers.MaxPooling2D(), # bloco 3: partes do objeto
        # resume cada mapa de características num único número (menos parâmetros)
        layers.GlobalAveragePooling2D(),
        # desliga 30% dos neurônios no treino, para reduzir overfitting
        layers.Dropout(0.3),
        # saída única: probabilidade de ser a classe 1 (tesoura)
        layers.Dense(1, activation='sigmoid'),
    ], name='cnn_do_zero')

with tf.device(TF_DEVICE):
    cnn = criar_cnn()
# otimizador Adam e perda de entropia cruzada binária (duas classes)
cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

# cronometra o treino da CNN
inicio = time.time()
# limita as threads do pipeline de dados
opcoes = tf.data.Options()
opcoes.threading.private_threadpool_size = 2
opcoes.threading.max_intra_op_parallelism = CPU_THREADS
# lotes de 8 imagens; o treino é embaralhado com semente fixa
dados_tr = tf.data.Dataset.from_tensor_slices((X_tr,y_tr)).shuffle(len(y_tr),seed=SEMENTE).batch(8).with_options(opcoes)
dados_va = tf.data.Dataset.from_tensor_slices((X_va,y_va)).batch(8).with_options(opcoes)
# até 80 épocas; para antes se a perda de validação não melhorar por 15 épocas
hist = cnn.fit(dados_tr, validation_data=dados_va, epochs=80, verbose=0,
               callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])
treino_cnn = (time.time() - inicio) / 60
epocas_cnn = len(hist.history['loss'])
# salva o modelo e o histórico do treino
cnn.save(str(Path(RUNS)/'cnn_do_zero.keras'))
pd.DataFrame(hist.history).to_csv(Path(RUNS)/'historico_cnn.csv', index=False)
print(f'Treino da CNN: {treino_cnn:.1f} min em {epocas_cnn} épocas'
      + (' (interrompido pelo early stopping)' if epocas_cnn < 80 else ' (todas as épocas)'))

# curvas de perda e acurácia: treino × validação
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, met, titulo in [(a1, 'loss', 'Perda'), (a2, 'accuracy', 'Acurácia')]:
    ax.plot(hist.history[met], color=COR_30, label='treino')
    ax.plot(hist.history[f'val_{met}'], color=COR_60, label='validação')
    ax.set_title(f'CNN do zero — {titulo}'); ax.set_xlabel('época'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# CNN: inclui leitura e resize como no pipeline YOLO; .numpy() aguarda o resultado.
# previsão da CNN numa imagem: abre, redimensiona para 128 px e classifica
def prever_cnn(p):
    with Image.open(p) as im:
        x = np.asarray(im.convert('RGB').resize((TAM,TAM)), dtype='float32')
    with tf.device('/CPU:0'):
        prob = float(cnn(x[None], training=False).numpy()[0,0])
    # acima de 0,5 = tesoura (classe 1); abaixo = xícara (classe 0)
    return int(prob > 0.5)
# mesmo protocolo de tempo usado nos dois YOLO
pred_cnn, inf_cnn = medir(prever_cnn, 'CNN do zero')
display(pd.DataFrame({'imagem':[Path(p).name for p in imgs_teste], 'real':nomes(y_te), 'CNN do zero':nomes(pred_cnn)}))

### 3.3 Quadro comparativo

In [ ]:
# Quadro comparativo da Entrega 2
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

def metricas(y, pred):
    # acurácia + precisão/recall/F1 médios das duas classes ("não detectou" conta como erro)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], average='macro', zero_division=0)
    return accuracy_score(y, pred), p, r, f

# previsões, tempo de treino e tempo de inferência de cada abordagem
resultados = {'YOLO customizado': (pred_custom, f'{tempos_treino.get(melhor_ep, np.nan):.1f} min', inf_custom),
              'YOLO padrão':      (pred_padrao, '0 (já vem pronto)', inf_padrao),
              'CNN do zero':      (pred_cnn, f'{treino_cnn:.1f} min', inf_cnn)}
# critérios qualitativos pedidos no enunciado: facilidade de uso e alcance
qualitativo = {'YOLO customizado': ('Média: exige rotular caixas', 'Sim', 'As que treinarmos'),
               'YOLO padrão':      ('Alta: pronto para usar', 'Sim', 'Só as 80 do COCO'),
               'CNN do zero':      ('Média: exige projetar a rede', 'Não', 'As que treinarmos')}

# monta o quadro: acertos, métricas, tempos (média, mediana, p95) e critérios
quadro = pd.DataFrame({nome: {
    'Acertos no teste': f'{sum(int(a == b) for a, b in zip(y_true, pred))}/{len(y_true)}',
    'Acurácia': metricas(y_true, pred)[0], 'Precisão (macro)': metricas(y_true, pred)[1],
    'Recall (macro)': metricas(y_true, pred)[2], 'F1 (macro)': metricas(y_true, pred)[3],
    'Tempo de treino/customização': treino, 'Inferência (ms/imagem)': inf,
    'Inferência mediana (ms)': float(np.median(latencias[nome])),
    'Inferência p95 (ms)': float(np.percentile(latencias[nome],95)),
    'Facilidade de uso': qualitativo[nome][0], 'Localiza o objeto?': qualitativo[nome][1],
    'Classes que reconhece': qualitativo[nome][2],
} for nome, (pred, treino, inf) in resultados.items()})
# salva o quadro e as previsões de cada modelo em CSV
quadro.to_csv(Path(RUNS)/'comparacao_modelos.csv')
pd.DataFrame({'imagem':[Path(p).name for p in imgs_teste], 'real':y_true, 'yolo_custom':pred_custom, 'yolo_padrao':pred_padrao, 'cnn':pred_cnn}).to_csv(Path(RUNS)/'predicoes_teste.csv',index=False)
display(quadro.style.format(lambda v: f'{v:.3f}' if isinstance(v, float) else v))

In [ ]:
# Exibe os gráficos no notebook.
%matplotlib inline
estilo_graficos()
# Acurácia e tempo de inferência em gráficos SEPARADOS (escalas diferentes não dividem eixo)
# valores usados nos dois gráficos de barras
nomes_modelos = list(resultados)
acuracias = [metricas(y_true, resultados[n][0])[0] for n in nomes_modelos]
inferencias = [resultados[n][2] for n in nomes_modelos]

# acurácia e tempo de inferência em gráficos separados
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, valores, titulo, fmt in [(a1, acuracias, 'Acurácia no teste (maior é melhor)', '{:.0%}'),
                                 (a2, inferencias, 'Tempo de inferência por imagem, ms (menor é melhor)', '{:.1f} ms')]:
    # uma barra por modelo, com o valor escrito em cima
    barras = ax.bar(nomes_modelos, valores, color=CORES_MODELOS, width=0.55, edgecolor='white', linewidth=2)
    ax.bar_label(barras, labels=[fmt.format(v) for v in valores], padding=4, fontsize=11)
    ax.set_title(titulo, fontsize=12); ax.grid(axis='x', visible=False)
    ax.set_ylim(0, max(valores) * 1.2 if max(valores) > 0 else 1)
a1.set_ylim(0, 1.15); a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.tight_layout(); plt.show()

# Matrizes de confusão: onde cada modelo erra
# uma matriz de confusão por modelo
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, nome in zip(axes, nomes_modelos):
    pred = resultados[nome][0]
    # inclui a coluna 'não detectou' quando houver abstenção
    rotulos = [0, 1, -1] if -1 in pred else [0, 1]
    exibicao = CLASSES + (['não detectou'] if -1 in pred else [])
    ConfusionMatrixDisplay(confusion_matrix(y_true, pred, labels=rotulos), display_labels=exibicao).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nome); ax.grid(False)
plt.tight_layout(); plt.show()

### 3.4 Análise de erros
As imagens abaixo permitem inspecionar os erros e levantar hipóteses sobre fundo, tamanho, ângulo ou iluminação. Uma inspeção visual não comprova a causa: seria necessário testar cada hipótese com novas imagens controladas. Se nenhum erro ocorrer, isso é informado sem fabricar exemplos.


In [ ]:
# Exibe os gráficos no notebook.
%matplotlib inline
estilo_graficos()
# Lista todas as imagens de teste em que algum modelo errou
erros_teste = [(nome, p, y, pr) for nome in nomes_modelos
               for p, y, pr in zip(imgs_teste, y_true, resultados[nome][0]) if y != pr]
# se ninguém errou, informa; senão, mostra cada erro com a classe real e a prevista
if not erros_teste:
    print('Nenhum dos três modelos errou nas imagens de teste.')
else:
    cols = 4; linhas_grade = int(np.ceil(len(erros_teste) / cols))
    fig, axes = plt.subplots(linhas_grade, cols, figsize=(16, 3.9 * linhas_grade), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, (nome, p, y, pr) in zip(axes.flat, erros_teste):
        ax.imshow(Image.open(p))
        ax.set_title(f'{nome}\n{os.path.basename(p)}: era {CLASSES[y]}, previu {CLASSES[pr] if pr >= 0 else "nada"}', fontsize=9)
    plt.suptitle('Erros no teste: o que cada modelo confundiu', fontsize=14); plt.tight_layout(); plt.show()
    display(pd.Series([e[0] for e in erros_teste]).value_counts().rename('imagens erradas').to_frame())

## 4. Conclusões
Resumo automático com os números desta execução:

In [ ]:
# Conclusões automáticas da Entrega 2 a partir dos números medidos
acc = dict(zip(nomes_modelos, acuracias)); inf = dict(zip(nomes_modelos, inferencias))
# modelos com a maior acurácia (empates são relatados como empates)
empatados = [n for n in nomes_modelos if acc[n] == max(acc.values())]
# modelo com a menor latência média
mais_rapido = min(inf, key=inf.get)
texto = [f"**Maior acurácia observada:** {', '.join(empatados)} ({max(acc.values()):.1%}). Empate é relatado como empate.",
         f"**Menor latência média neste protocolo:** {mais_rapido}, {inf[mais_rapido]:.1f} ms/imagem.",
         f"**Abstenções do YOLO padrão:** {sum(p == -1 for p in pred_padrao)}/{len(pred_padrao)}; contabilizadas como erros.",
         '**Tempo real:** o inverso da latência é apenas uma estimativa de capacidade. Captura, transmissão, renderização e estabilidade precisam ser medidos em vídeo real.']
# intervalo de confiança de Wilson (95%) para a taxa de acertos em 8 imagens
def wilson(k,n,z=1.96):
    p = k/n; den = 1+z*z/n
    meio = (p+z*z/(2*n))/den
    margem = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return meio-margem, meio+margem
# uma frase por modelo com acertos e intervalo de confiança
for nome in nomes_modelos:
    k = sum(int(a == b) for a,b in zip(y_true,resultados[nome][0]))
    lo,hi = wilson(k,len(y_true))
    texto.append(f'**{nome}:** {k}/{len(y_true)} acertos; intervalo de Wilson 95% ilustrativo [{lo:.1%}, {hi:.1%}].')
texto.append('Os intervalos pressupõem observações independentes; fotos correlacionadas do mesmo ambiente reduzem essa interpretação. Não são intervalos de mAP.')
display(Markdown('\n\n'.join('- '+x for x in texto)))

In [ ]:
# Leitura dos números da Entrega 2, gerada a partir das medições DESTA execução (nenhum valor é digitado à mão)
br = lambda v, c=1: f'{v:.{c}f}'.replace('.', ',')    # número no padrão brasileiro (vírgula decimal)
n_teste = len(y_true)
acertos = {n: sum(int(a == b) for a, b in zip(y_true, resultados[n][0])) for n in nomes_modelos}
if len(set(acertos.values())) == 1:
    s_acc = (f'Os {len(nomes_modelos)} modelos acertaram {list(acertos.values())[0]}/{n_teste} nas imagens de teste; '
             'logo, a acurácia não os diferencia e o que separa as abordagens é custo e capacidade.')
else:
    lista = '; '.join(f'{n} {k}/{n_teste}' for n, k in acertos.items())
    s_acc = (f'Acertos no teste: {lista}. Com apenas {n_teste} imagens, cada erro vale {br(100 / n_teste)} pontos percentuais, '
             'mas a acurácia já ajuda a separar as abordagens.')
ordem = sorted(nomes_modelos, key=inf.get)            # do mais rápido ao mais lento
s_inf = ('Na inferência em CPU (da leitura da imagem até a decisão, batch 1), do mais rápido ao mais lento: '
         + ', '.join(f'{n} (cerca de {inf[n]:.0f} ms/imagem)' for n in ordem) + '.')
razao = round(inf['YOLO padrão'], 3) / round(inf['YOLO customizado'], 3)
s_inf += (f' O YOLOv3 padrão levou cerca de {br(razao)}× o tempo do YOLOv5s customizado, '
          'ressalvadas as diferenças de arquitetura, backend e resolução.')
s_trn = (f'Na customização, a CNN treinou em cerca de {br(treino_cnn)} min, o YOLOv5s selecionado ({melhor_ep} épocas) em cerca de '
         f'{br(tempos_treino[melhor_ep])} min e o YOLOv3 padrão não exigiu treino. Apenas os dois YOLO localizam o objeto.')
display(Markdown('**Leitura dos números** (gerada automaticamente a partir desta execução). ' + ' '.join([s_acc, s_inf, s_trn])))


**Facilidade de integração.** YOLO padrão dispensa treinamento próprio, mas precisa ter classes compatíveis com o problema. YOLO customizado exige coleta, caixas e ajuste; a CNN do zero exige arquitetura e rótulos de imagem, dispensando caixas para seu treinamento. Para comparar as três abordagens usamos as mesmas fotos e derivamos a classe das anotações existentes.

**Precisão e localização.** O quadro anterior mede classificação por imagem; mAP da seção 2.4 mede detecção. A CNN não produz caixas, por isso não recebe mAP e não substitui o detector quando a posição do objeto é necessária. Uma abstenção após filtrar classes COCO é um erro operacional, não evidência de uma classe específica errada.

**Custos.** No YOLO, cronometramos o processo de treinamento completo; na CNN, a chamada `fit`, sem construção da rede. Esses tempos têm limites distintos e representam custo prático aproximado; coleta e rotulação não foram cronometradas. O YOLO padrão tem custo de customização zero neste experimento, mas seu pré-treinamento não foi gratuito. A latência depende de hardware, resolução e implementação. Comparar média, mediana e p95 evita depender de uma única chamada.

**Decisão.** Para reconhecer **e localizar** xícara e tesoura, o YOLO customizado é a opção alinhada ao requisito, condicionado à validação em novos ambientes. Para apenas classificar uma foto, a CNN pode ser suficiente. Não há vencedor universal.

**Limitações.** 64 imagens de treino e 8 de teste; uma semente nos treinos obrigatórios; ambientes possivelmente correlacionados; ausência de casos negativos e de avaliação em cenas com múltiplos objetos. Uma acurácia alta em apenas 8 imagens de teste (cada imagem vale 12,5 pontos percentuais) não é garantia de generalização. Não se conclui overfitting apenas de uma pequena queda de mAP.

## 5. Ficha técnica e resultados
A ficha registra ambiente e parâmetros. Cada execução salva métricas, predições, gráficos e pesos em sua pasta de resultados. Os arquivos do experimento apresentado acompanham o projeto em `assets/`.


In [ ]:
import platform, json, datetime
# Ficha técnica: tudo o que é preciso para repetir o experimento
br = lambda v, casas=3: f'{v:.{casas}f}'.replace('.', ',')             # número no padrão brasileiro (vírgula decimal)
# versão exata do YOLOv5 usada
commit = subprocess.run(['git', '-C', YOLO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ficha = {
    'Data da execução': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'),
    'Hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'Python': platform.python_version(), 'PyTorch': torch.__version__, 'TensorFlow': tf.__version__,
    'YOLOv5 (commit)': commit, 'Pesos iniciais': 'yolov5s.pt (COCO)',
    'Épocas comparadas': ' × '.join(map(str, EPOCAS)), 'Imagem (px)': IMG_SIZE, 'Batch': BATCH,
    'Semente': SEMENTE, 'Confiança mínima': CONF, 'Classes': ', '.join(CLASSES),
    'TensorFlow device': TF_DEVICE, 'Inferência comparativa': 'CPU, todos os modelos', 'Referência padrão': 'YOLOv3 COCO / OpenCV DNN', 'Protocolo latência': 'disco até classe, batch=1, 3 warmups, 5 repetições',
    'Imagens treino / val / teste': ' / '.join(str(len(listar_imagens(s))) for s in ['train', 'val', 'test']),
}
display(pd.Series(ficha, name='valor').astype(str).to_frame())

# junta ficha técnica, métricas e tempos num único arquivo de resultados
consolidado = {'ficha_tecnica': ficha, 'entrega1': tabela_avaliacao.reset_index().to_dict('records'),
               'tempos_treino_min': tempos_treino, 'melhor_epocas': melhor_ep,
               'entrega2': {n: {'acuracia': acc[n], 'inferencia_ms': inf[n]} for n in nomes_modelos},
               'treino_cnn_min': treino_cnn, 'latencias_ms':latencias}
# registra a versão de todas as bibliotecas instaladas
subprocess.run([sys.executable,'-m','pip','freeze'], stdout=open(str(Path(RUNS)/'ambiente_pip_freeze.txt'),'w'), check=True)
# grava os resultados consolidados em JSON
with open(f'{RUNS}/resultados.json', 'w', encoding='utf-8') as f:
    json.dump(consolidado, f, ensure_ascii=False, indent=2, default=str)

print('Resultados salvos em:', RUNS)

### Referências
- FIAP — Fase 6: enunciado do projeto e materiais de Redes Neurais e Visão Computacional.
- FIAP — Material de Redes Neurais e Visão Computacional: YOLO tradicional (YOLOv3/Darknet), conforme indicado no enunciado da Fase 6.
- [Ultralytics YOLOv5](https://github.com/ultralytics/yolov5).
- [Darknet / YOLOv3](https://github.com/pjreddie/darknet).
- [Pesos YOLOv3](https://github.com/AlexeyAB/darknet/releases/tag/darknet_yolo_v3_optimal).
- [Make Sense AI](https://www.makesense.ai/).
- Lin et al. (2014). *Microsoft COCO: Common Objects in Context*.
